# 05c — 24-hour landmark severity sensitivity

## Summary

Explore first-day severity in admissions alive and still hospitalized beyond 24 hours after ICU entry. Reset follow-up at that landmark and compare age/sex and SOFA-core models on the same complete cases. This changes the population and estimand; default results are synthetic.

## Context & Methods

Fix the landmark at 24 elapsed hours from the original ICU entry. Exclude death/discharge endpoints at or before it. Also exclude remaining deaths with missing death timestamps because their true survival to the landmark is unknown. Preserve the original pre-ICU prescription-order classification, subtract one day from eligible durations, and use era strata and patient-cluster Efron Cox inference.

### Key assumptions
- The published `first_day_sofa` definition uses measurements through ICU entry plus one day. The deployed component windows and individual measurement times have not been independently reconstructed.
- This is a conditional landmark sensitivity analysis, not a reconstruction of pre-ICU baseline severity.
- First-day SOFA follows exposure and can mediate associations or introduce selection/collider bias. Landmarking does not establish causality.
- Keep CHF, CKD and diabetes definitions from the original core model; their diagnosis timing is not newly validated.
- Use identical core-complete admissions for both landmark fits; report excluded early endpoints and missingness.

## Data

Run from the repository root or `notebooks/` after `uv sync --locked`. The existing bounded BigQuery loader is used only for explicit `current` mode. Pseudonymous keys stay in memory; never print source rows. Use `local` mode to inspect a previously verified aggregate run without another query.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from survival_sensitivity import run_supplement, load_local_bundle, plot_supplement
%matplotlib inline
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})

In [ ]:
# Default: artificial data only. Never display a patient-level DataFrame.
DATA_MODE = "synthetic"  # "synthetic", "current", or "local"
LOCAL_RESULT = REPO_ROOT / "reports/local_supplements/sensitivity_run02/aggregate_results.json"
BOOTSTRAP_REPLICATES = 1000 if DATA_MODE == "current" else 200

if DATA_MODE == "local":
    bundle = load_local_bundle(LOCAL_RESULT, REPO_ROOT, "landmark")
elif DATA_MODE in {"synthetic", "current"}:
    bundle = run_supplement("landmark", repo=REPO_ROOT, clinical=DATA_MODE == "current",
                            bootstrap_replicates=BOOTSTRAP_REPLICATES)
else:
    raise ValueError("Unknown DATA_MODE")
result = bundle["results"]["landmark"]
print({"data_origin": bundle["data_origin"], "supplement_version": bundle["supplement_version"],
       "main_analysis_updated": bundle["main_analysis_updated"]})

## Results

The data-origin label above controls interpretation. Synthetic outputs validate execution only.

### Eligibility and exclusions

In [ ]:
display(pd.DataFrame(result["eligibility"].items(), columns=["Eligibility measure", "Admissions / days"]))

### Matched landmark models

In [ ]:
rows = []
for name, model in result["models"].items():
    e = model["estimates"]["raas_pre_icu"]
    rows.append({"Model": name, "Admissions": model["n"], "Patients": model["patients"],
                 "RAAS HR": e["hr"],
                 "Ordinary 95% CI": f"{e['ordinary']['lower_95']:.3f}–{e['ordinary']['upper_95']:.3f}",
                 "Cluster 95% CI": f"{e['patient_cluster']['lower_95']:.3f}–{e['patient_cluster']['upper_95']:.3f}",
                 "Ordinary p": e["ordinary"]["p"], "Cluster p": e["patient_cluster"]["p"]})
display(pd.DataFrame(rows).style.format({"RAAS HR": "{:.3f}", "Ordinary p": "{:.4g}", "Cluster p": "{:.4g}"}))

### Visual comparison

In [ ]:
ANALYSIS = "landmark"
figure = plot_supplement(bundle, ANALYSIS)
display(figure)
plt.close(figure)

## Takeaways

Compare the two landmark models within their matched eligible cohort. Differences from the original ICU-entry analysis combine selection, a later risk origin and adjustment changes; they cannot isolate a baseline severity effect. A pre-ICU measurement-based severity design remains a separate analysis.

See [the supplement guide](../docs/SURVIVAL_SENSITIVITY_SUPPLEMENTS.md) for the execution commands, data handling, reference checks and limitations. Tracked notebook outputs remain stripped; keep executed clinical copies under ignored `reports/local_supplements/`.